# 04 — Modelagem de Anomalias (Não-Supervisionada)

Este notebook documenta a etapa de **detecção de anomalias não-supervisionada** usando Isolation Forest.

## Pipeline:
1. Carregamento do dataset analítico com scores do IF (já calculados em `src/anomaly.py`)
2. Análise exploratória dos scores
3. Comparação com candidatos heurísticos da EDA
4. Preparação de `anomalia_score` como "verdade" para etapa supervisionada (próximo notebook)

**Nota importante:** não estamos "predizendo" algo aqui — estamos **documentando os padrões descobertos** pelo IF. A próxima etapa usará esses scores como target para treinar modelos supervisionados interpretáveis.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import INTERIM_DIR

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

## Carregamento do dataset com scores

In [ ]:
# Carregar dataset com scores do IF (gerado em src/anomaly.py)
path = INTERIM_DIR / "dataset_analitico_com_scores.parquet"
df = pd.read_parquet(path)

print(f"Dataset: {df.shape}")
print(f"\nColunas principais:")
print(f"  - anomalia_score: {df['anomalia_score'].dtype}")
print(f"  - anomalia_label: {df['anomalia_label'].dtype}")
print(f"  - fold: {df['fold'].value_counts().to_dict()}")
print(f"\nAmomalias por fold:")
print(df.groupby('fold')['anomalia_label'].agg(['count', 'sum', 'mean']))

## Análise exploratória dos scores

In [ ]:
# Estatísticas gerais
print("Estatísticas de anomalia_score:")
print(df['anomalia_score'].describe())

# Por fold
print("\nPor fold (treino vs teste):")
for fold in ['train', 'test']:
    s = df[df['fold'] == fold]['anomalia_score']
    print(f"\n{fold.upper()}:")
    print(f"  Mean: {s.mean():.4f}, Std: {s.std():.4f}")
    print(f"  Min: {s.min():.4f}, Max: {s.max():.4f}")
    print(f"  p25: {s.quantile(0.25):.4f}, p50: {s.quantile(0.5):.4f}, p75: {s.quantile(0.75):.4f}")

In [ ]:
# Distribuição dos scores
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
df['anomalia_score'].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Distribuição de anomalia_score (todos os dados)')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequência')

# Box plot por fold
df.boxplot(column='anomalia_score', by='fold', ax=axes[1])
axes[1].set_title('anomalia_score por fold (treino vs teste)')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

## Comparação: IF vs. Heurísticas da EDA

In [ ]:
# Comparar anomalia_label (IF binário) com candidato_anomalia_heuristica
df['heuristica'] = df['candidato_anomalia_heuristica'].astype(int)
df['if_label'] = df['anomalia_label'].astype(int)

# Matriz de confusão
print("Comparação: Heurística vs. Isolation Forest")
print("\nTabela cruzada (heuristica vs IF):")
tab = pd.crosstab(df['heuristica'], df['if_label'], margins=True, margins_name='Total')
tab.index = ['Heuristica=0', 'Heuristica=1', 'Total']
tab.columns = ['IF=0', 'IF=1', 'Total']
print(tab)

# Estatísticas de overlap
ambos = ((df['heuristica'] == 1) & (df['if_label'] == 1)).sum()
apenas_heur = ((df['heuristica'] == 1) & (df['if_label'] == 0)).sum()
apenas_if = ((df['heuristica'] == 0) & (df['if_label'] == 1)).sum()

print(f"\nAnálise de overlap:")
print(f"  Ambos identificam como anomalia: {ambos:,}")
print(f"  Apenas heurística: {apenas_heur:,}")
print(f"  Apenas IF: {apenas_if:,}")
print(f"  Concordância: {(ambos + ((df['heuristica'] == 0) & (df['if_label'] == 0)).sum()) / len(df):.2%}")

## Top anomalias detectadas pelo IF

In [ ]:
# Top 20 anomalias
top_anom = df.nlargest(20, 'anomalia_score')[[
    'numero_licitacao', 'modalidade_compra', 'valor_licitacao', 
    'n_participantes', 'anomalia_score', 'candidato_anomalia_heuristica'
]]

print("Top 20 anomalias por anomalia_score:")
print(top_anom.to_string())

## Correlação entre anomalia_score e features

In [ ]:
# Features numéricas usadas no IF
feats_if = ['log_valor_licitacao', 'n_participantes', 'n_itens', 'hhi', 'top1_share', 'valor_total_itens']

# Correlação com anomalia_score
corr = df[feats_if + ['anomalia_score']].corr()['anomalia_score'].sort_values(ascending=False)

print("Correlação com anomalia_score:")
print(corr)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 4))
corr[:-1].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Correlação entre features do IF e anomalia_score')
ax.set_xlabel('Correlação')
plt.tight_layout()
plt.show()

## Resumo e próximos passos

### Achados:
- **Distribução de scores:** [inserir observações do output acima]
- **Overlap com heurísticas:** [% de concordância, diferenças observadas]
- **Features mais correlacionadas:** [quais features direcionam o score]

### Próximo passo:
No notebook `05_modelagem_supervisionada.ipynb`, vamos usar `anomalia_score` como **label contínuo** (ou binarizado em threshold) para treinar modelos supervisionados interpretáveis (KNN, Árvore de Decisão, Random Forest). Esses modelos tentarão "aprender" os padrões que o IF detectou, mas de forma explicável.